# 04 - Keras Sequential Classifier

The paper reports approximately **99.21% accuracy** for the Keras Sequential classifier.

The paper reports approximately **99.21%** for the Keras Sequential model and Table 1.3 rounds it to `.992`.

The model setup used in the project includes:

- the large 80,207-row feature-table pipeline;
- 218 fine-grained target labels;
- 57 numerical input features;
- an 80/20 split;
- MinMax scaling;
- a dense Sequential code cell with hidden layers 512 -> 256 -> 128 -> 64, ReLU and 0.2 dropout;
- Adam, sparse categorical cross-entropy, batch size 128, and a 1500-epoch training call.

The original saved 99.21% checkpoint is no longer available, so the following details cannot be reproduced exactly:

- the exact final working output-layer code;
- the exact train/test split;
- random initialization;
- trained weights/checkpoint;
- the saved 99.21% evaluation output.

The cleaned implementation uses a dynamic `n_classes` output layer so it matches the classification label count.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

sys.path.append("..")
from src.labels import fine_label_ids_in_original_order

DATA_PATH = Path("../data/new_features_3sec.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The recovered 80,207-row new_features_3sec.csv is missing. "
        "See data/README.md and notebook 01 to regenerate it from the original audio."
    )

data = pd.read_csv(DATA_PATH)
y, mapping = fine_label_ids_in_original_order(data["filename"])
X_df = data.drop(columns=["filename", "length"])

print("Table shape:", data.shape)
print("Numeric features:", X_df.shape[1])
print("Fine-grained labels:", len(mapping))

In [ ]:
# Historical preprocessing preserved from the recovered notebook:
# MinMaxScaler was fit before the split and no random_state was stored.
scaler = MinMaxScaler()
X = scaler.fit_transform(X_df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
from src.deep_models import build_reconstructed_sequential

n_features = X_train.shape[1]
n_classes = len(mapping)

model = build_reconstructed_sequential(n_features, n_classes)
model.summary()

## Training

The experiment uses **1500 epochs** with **batch size 128**.

This is expensive and intentionally not pre-executed.

In [ ]:
EPOCHS = 1500
BATCH_SIZE = 128

# history = model.fit(
#     X_train,
#     y_train,
#     validation_data=(X_test, y_test),
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
# )

## Evaluation

The paper-reported reference accuracy is shown below. A newly trained model should report its own measured accuracy.

In [ ]:
PAPER_REPORTED = 0.9921
print(f"Paper-reported reference: {PAPER_REPORTED:.2%}")

# test_loss, test_accuracy = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE)
# print(f"New reproduction test accuracy: {test_accuracy:.4%}")

## Cleaner modern protocol

For a cleaner modern evaluation, split first, fit the scaler only on the training set, use a fixed seed, and keep a separate untouched test set.